In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd #to store data as csv or excel format
import time
from datetime import date

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.janabank.com/interest-rates/#retail-fixed-deposits-for-domestic-nro-nre-customers" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Provide the path to your ChromeDriver
service = Service(CHROME_DRIVER_PATH)

# Initialize the WebDriver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)

try:
    # Open the webpage
    driver.get(URL)

    # Wait until the content is loaded
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

    # Give it extra time if needed
    time.sleep(5)

    # Get the page source after rendering JavaScript
    page_content = driver.page_source

    # You can parse this with BeautifulSoup if needed
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(page_content, 'html.parser')
    table=soup.find_all("table")
    fd_table=table[2]
    rows=fd_table.find_all("tr")
    
    print("Today's date =",date1)
    print("Interest rate file name = ",file_name)
    
    list_of_interest_rates = []
    tax_saver_row = []
    row_count = 0
    for i in rows[1:]:  #Skip heading
        data = i.find_all("td")
        #print(data)
        row=['Jana SF Bank']
        row.extend([tr.text for tr in data[:1]])
        row.extend([tr.text for tr in data[3:]])
        row.insert(3,'')
        row.extend([''])
        if row_count == 9: 
            tax_saver_row = ['Jana SF Bank', 'Tax-saver Fixed Deposit', row[2], '',row[4], '']
            #for i in range(0,9):
            #    tax_saver_row[i] = ' '.join(str(tax_saver_row[i]).strip().replace('\n', '').split())
            list_of_interest_rates.append(tax_saver_row)
        list_of_interest_rates.append(row)
        row_count += 1
        
    #print(list_of_interest_rates)   
    pd.set_option('display.max_columns', None)    
    int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                            'sr_rate','annualised_sr_rate'])
    print(int_rate_df)
    int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
    print("\nJana SF Bank = Success. Number of rows added = ", len(list_of_interest_rates))
finally:
    # Close the driver
    driver.quit()

Today's date = 12012025
Interest rate file name =  c:\Interest_rate_files\Interest_rate_12012025.csv
       bank_name                          tenure gen_rate annualised_gen_rate  \
0   Jana SF Bank                       7-14 Days    6.75%                       
1   Jana SF Bank                     15-60 Days     6.75%                       
2   Jana SF Bank                      61-90 Days    6.75%                       
3   Jana SF Bank                     91-180 Days    6.75%                       
4   Jana SF Bank                    181-364 Days   7.50%                        
5   Jana SF Bank                       365 Days     8.25%                       
6   Jana SF Bank     1 Year - 2 Years(730 Days)     8.25%                       
7   Jana SF Bank  > 2 Years - 3 Years(1095 Days     8.25%                       
8   Jana SF Bank            > 3 Year - < 5 Years    7.25%                       
9   Jana SF Bank         Tax-saver Fixed Deposit    8.20%                       
10  Jana